In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key="sk-762684b96deb4f748cb4383757f69a09",model="deepseek-chat" ,base_url="https://api.deepseek.com") 

## Chain & Pipe

In [ ]:
Chain & Pipe

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
chain = prompt | llm | StrOutputParser()

analysis_prompt = ChatPromptTemplate.from_template("is this a funny joke? {joke}")
composed_chain_with_lambda = (
    chain
    | (lambda input: {"joke": input})
    | analysis_prompt
    | llm
    | StrOutputParser()
)

composed_chain_with_lambda.invoke({"topic": "beets"})

'Haha, that\'s a classic! It\'s a lighthearted and punny joke, so it definitely qualifies as funny, especially if you enjoy a good play on words. The idea of a beet "tressing" (dressing) up and turning red is clever and cute. If you like veggie humor, this one\'s a winner! 😄'

In [4]:
chain.invoke({"topic": "beets"})

"Sure! Here's a beet-related joke for you:\n\nWhy did the beet turn red?\n\nBecause it saw the salad dressing! 🥗😄"

In [5]:
from langchain_core.runnables import RunnableParallel

composed_chain_with_pipe = (
    RunnableParallel({"joke": chain})
    .pipe(analysis_prompt)
    .pipe(llm)
    .pipe(StrOutputParser())
)

composed_chain_with_pipe.invoke({"topic": "battlestar galactica"})

'Yes, that\'s a funny joke! It’s a clever play on words, combining the idea of a Cylon (a robotic race from *Battlestar Galactica*) having "issues" with its programming, which could mean both technical problems and emotional struggles. The light-hearted tone and the nod to the show\'s fandom make it enjoyable, especially for fans of the series. Plus, the "frakking around" pun at the end is a nice touch! 😄'

## Formatting with RunnableParallels

RunnableParallels are useful for parallelizing operations, but can also be useful for manipulating the output of one Runnable to match the input format of the next Runnable in a sequence. You can use them to split or fork the chain so that multiple components can process the input in parallel. Later, other components can join or merge the results to synthesize a final response. This type of chain creates a computation graph that looks like the following:

       Input
        / \
       /   \
 Branch1 Branch2

       \   /
        \ /
        Combine

 {"context": retriever, "question": RunnablePassthrough()}

RunnableParallel({"context": retriever, "question": RunnablePassthrough()})

RunnableParallel(context=retriever, question=RunnablePassthrough())

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings

vectorstore = FAISS.from_texts(
    ["harrison worked at kensho"], embedding=HuggingFaceEmbeddings()
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# The prompt expects input with keys for "context" and "question"
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(api_key="sk-762684b96deb4f748cb4383757f69a09",model="deepseek-chat" ,base_url="https://api.deepseek.com")

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()} # get the context using our retriever and passthrough the user input under the "question" key
    | prompt #
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("where did harrison work?")

'Based on the provided context, Harrison worked at Kensho.'

## Using itemgetter as shorthand

Note that you can use Python's itemgetter as shorthand to extract data from the map when combining with RunnableParallel. You can find more information about itemgetter in the Python Documentation.

In [9]:
from operator import itemgetter

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke({"question": "where did harrison work", "language": "italian"})

'Harrison worked at Kensho.'

## Parallelize steps

RunnableParallels make it easy to execute multiple Runnables in parallel, and to return the output of these Runnables as a map.##

In [10]:
joke_chain = ChatPromptTemplate.from_template("tell me a joke about {topic}") | model
poem_chain = (
    ChatPromptTemplate.from_template("write a 2-line poem about {topic}") | model
)

map_chain = RunnableParallel(joke=joke_chain, poem=poem_chain)

map_chain.invoke({"topic": "bear"})

{'joke': AIMessage(content="Sure, here's a bear joke for you:\n\nWhy did the bear bring a ladder to the bar?\n\nBecause he heard the drinks were on the house! 🐻🍻", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 9, 'total_tokens': 45, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-a05cb72b-f864-4530-9568-32d44fa160da-0', usage_metadata={'input_tokens': 9, 'output_tokens': 36, 'total_tokens': 45, 'input_token_details': {}, 'output_token_details': {}}),
 'poem': AIMessage(content='In the forest deep, the bear roams free,  \nA shadow of strength beneath the tree.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 11, 'total_tokens': 31, 'completion_tokens_detai

## Parallelism

RunnableParallel are also useful for running independent processes in parallel, since each Runnable in the map is executed in parallel. For example, we can see our earlier joke_chain, poem_chain and map_chain all have about the same runtime, even though map_chain executes both of the other two.

In [11]:
%%timeit

joke_chain.invoke({"topic": "bear"})

1.82 s ± 147 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit

poem_chain.invoke({"topic": "bear"})

1.39 s ± 125 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit

map_chain.invoke({"topic": "bear"})

1.68 s ± 177 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## add default invocation args to a Runnable
Sometimes we want to invoke a Runnable within a RunnableSequence with constant arguments that are not part of the output of the preceding Runnable in the sequence, and which are not part of the user input. We can use the Runnable.bind() method to set these arguments ahead of time.




In [15]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write out the following equation using algebraic symbols then solve it. Use the format\n\nEQUATION:...\nSOLUTION:...\n\n",
        ),
        ("human", "{equation_statement}"),
    ]
)

runnable = (
    {"equation_statement": RunnablePassthrough()} | prompt | model | StrOutputParser()
)

print(runnable.invoke("x raised to the third plus seven equals 12"))

EQUATION: \( x^3 + 7 = 12 \)

SOLUTION:
1. Subtract 7 from both sides:
   \[
   x^3 = 12 - 7
   \]
   \[
   x^3 = 5
   \]
2. Take the cube root of both sides:
   \[
   x = \sqrt[3]{5}
   \]
   \[
   x \approx 1.71
   \]

So, the solution is \( x = \sqrt[3]{5} \).


In [16]:
runnable = (
    {"equation_statement": RunnablePassthrough()}
    | prompt
    | model.bind(stop="SOLUTION") #model.bind() add other tuntime args.
    | StrOutputParser()
)

print(runnable.invoke("x raised to the third plus seven equals 12"))

EQUATION: \( x^3 + 7 = 12 \)




In [17]:
from langchain_core.runnables import RunnableLambda

def length_function(text):
    return len(text)

def _multiple_length_function(text1, text2):
    return len(text1) * len(text2)

def multiple_length_function(_dict):
    return _multiple_length_function(_dict["text1"], _dict["text2"])


prompt = ChatPromptTemplate.from_template("what is {a} + {b}")
chain1 = prompt | model

chain = (
    {
        "a": itemgetter("foo") | RunnableLambda(length_function), # single input lambda
        "b": {"text1": itemgetter("foo"), "text2": itemgetter("bar")} # multiple input lambda
        | RunnableLambda(multiple_length_function),
    }
    | prompt
    | model
)

chain.invoke({"foo": "bar", "bar": "gah"})

AIMessage(content='**Solution:**\n\nTo find the sum of 3 and 9, follow these steps:\n\n1. **Identify the numbers to add:**\n   \\[\n   3 \\quad \\text{and} \\quad 9\n   \\]\n\n2. **Add the numbers together:**\n   \\[\n   3 + 9 = 12\n   \\]\n\n3. **Final Answer:**\n   \\[\n   \\boxed{12}\n   \\]', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 10, 'total_tokens': 102, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 10}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-3e99f62d-c4d9-4c37-bc01-9f6bb8574f1b-0', usage_metadata={'input_tokens': 10, 'output_tokens': 92, 'total_tokens': 102, 'input_token_details': {}, 'output_token_details': {}})